# OpenMOSS MOSS-Image (5B DiT) Cloud LoRA Training Pipeline

一套专为 **OpenMOSS MOSS-Image (5B Single-Stream DiT)** 打造的 Kaggle 云端 LoRA 自动化极速炼丹环境（当前适配管线 **v1.8.3**，全面同步本地桌面端：支持 **GPU T4 x2 双卡 DDP 并行加速**与目标算力预设自适应校准、**免排队交互式 Notebook 直推**、`--no-run` 绕过后台队列与交互式 GPU 运行、5 站外链直拉容灾 `DATASET_URL`、Hugging Face 私有仓库一键上传与直拉 `HF_DATASET_REPO`、Cloudflare R2 私有桶 7 天预签名直拉、超时自动存盘保护 `MAX_RUN_MINUTES`、LoKr 全参 / 三层 Dropout / TrainNorms / Muon 优化器与 EMA、多时间步采样、896 甜点分辨率、多子集配比与 `subsets_cloud.json` 自动识别、云端非阻断图集健康体检、DoRA / LoHa / LoRA+、全谱系 Base / Heal / SFT，以及 `metrics.jsonl` 曲线归档供本地桌面 GUI 复盘）。

### ⚡ 运行前必备设置 (请点击上方菜单栏中的 Settings)：
1. **Accelerator**: 推荐选择 **GPU T4 x2** (16GB × 2，自动启用 Dual-T4 DDP + `int8` `ckpt-resident` + `MAX_TOKENS=9216`，并发 4 张 1024 图/步提速近一倍；亦可兼容 **GPU T4 x1** / P100 / L4)；
2. **Internet**: 必须切换为 **Internet on** (用于极速拉取底模与依赖)；
3. **Data/Input**: 点击顶部 File -> Add Data 或右侧面板挂载您上传的训练图集。

In [ ]:
# 1. 验证 GPU 硬件算力与 PyTorch 原生驱动
import torch, torchvision
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("❌ 警告：未检测到 GPU！请在上方菜单栏 Settings -> Accelerator 中开启 GPU T4 x1。")

### 🎯 用户自定义超参数设置区
在此处根据您的数据集类型（画风/人物）修改对应的参数配置与触发词：

In [ ]:
# ==============================================================================
# ⚙️ 超参数控制面板 (修改下方变量即可完成个性化定制)
# ==============================================================================
LORA_NAME = "my_character"          # LoRA 产出英文名称 (如 makima, genshin_style)
TRIGGER_WORD = "my_character_char"  # 触发词 (例如: makima_char；留空 "" 则使用原标注)

REPEATS = 3                         # 单图重复次数 (1024 单卡画风推荐 3~4，Dual-T4 或角色配 GradAcc=2 推荐 6~10)
EPOCHS = 5                          # 训练轮数 (人物推荐 3~5，画风推荐 4~6)
STEPS = 400                         # 核心步数 (公式: 图片数 * REPEATS * EPOCHS // 真实等效Batch)

RANK = 32                           # LoRA 矩阵秩 (单概念推荐 32，多人物/多画风推荐 48~64)
ALPHA = 16.0                        # 缩放系数 (人物推荐 16.0，画风推荐 32.0)
LR = "1e-4"                         # 基础学习率 (Base/Heal 推荐 1e-4，SFT 画风推荐 1.2e-4~1.5e-4)
GRAD_ACC = 1                        # 梯度累积步数 (单画风=1 极速；角色细节/多子集混合=2)
TIMESTEP = "moss"                   # 时间步分布: "moss"(官方) | "uniform" | "logit_normal" (注意 logit_normal 正负号与 moss 相反!)
TIMESTEP_MU = -1.09                 # 时间步采样重心 Mu (moss下: -1.09=画风默认；-0.60=多画风；0.0=动作概念；0.35=角色五官细节)

# --- 进阶适配器架构与防过拟合/优化器控制 (v1.2 ~ v1.5) ---
ALGO = "lora"                       # 适配器算法: "lora"(默认通用) | "loha"(哈达玛高容量) | "lokr"(克罗内克超小体积)
LOKR_FACTOR = 8                     # LoKr 压缩因子 (-1=自动 | 4=高容量 | 8=默认 | 16=超紧凑)
LOKR_FULL_MATRIX = False            # LoKr 是否开启满秩第二矩阵 (提升拟真度但体积变大)
LOKR_DECOMPOSE_BOTH = False         # LoKr 是否同时分解双因子矩阵
TRAIN_NORMS = False                 # 是否同步训练模型内部的 AdaLN/RMS/LayerNorm 归一化层
USE_DORA = False                    # 是否启用 DoRA 权重分解 (人物/多画风/细节还原强推 True)
RANK_DROPOUT = 0.0                  # 防过拟合 Rank Dropout (小图集角色建议 0.05~0.1，0.0=关闭)
MODULE_DROPOUT = 0.0                # 随机跳过整个模块比例 (0.0~0.1，防背景死记)
INPUT_DROPOUT = 0.0                 # 输入随机丢弃 (开启 USE_DORA=True 时会自动锁定为 0.0)
TARGET_PRESET = "attn-mlp"          # 目标注入层: "attn" | "attn-mlp" | "attn-mlp+modulation"(多画风推荐) | "attn-mlp+text-refiner"(多人物/概念推荐) | "full-block"
LORAPLUS_RATIO = 1.0                # LoRA+ 上投影学习率倍率 (1.0=关闭，人物/多概念建议 2.0~4.0 加速收敛)
OPTIMIZER = "adamw"                 # 优化器: "adamw" | "adamw8bit_kernel" | "muon" (非 adamw 自动切安全流式档位)
LR_SCHEDULER = "cosine"             # 学习率曲线: "cosine"(余弦退火) | "constant"(恒定学习率)
WARMUP_STEPS = -1                   # 预热步数 (-1=自动按总步数 5% 计算)
WEIGHT_DECAY = 0.01                 # 权重衰减系数
WD_RULES = "moss"                   # 权重衰减规则: "moss" | "none" | "all"
EMA = 0.0                           # EMA 指数滑动平均平滑 (0.0=关闭；长步数或高学习率防抖建议 0.99~0.995)
TEXT_DROP = 0.05                    # 空提示词无条件丢弃概率 (默认 0.05，防过拟合可设 0.1)
FLIP = False                        # 水平镜像翻转 (人物非对称服饰/带文字图像严禁设为 True!)
SEED = 0                            # 随机种子
SAVE_FORMAT = "both"                # 权重导出格式: "both"(ComfyUI+LyCORIS) | "comfy" | "lycoris"
# 多子集配比 JSON (留空 "" 使用单图集；填写列表开启多人物/多画风独立 Repeat 与独立触发词):
# 示例: SUBSETS_JSON = '[{"name":"char_a","path":"/kaggle/input/set/a","trigger":"char_a","repeats":6},{"name":"char_b","path":"/kaggle/input/set/b","trigger":"char_b","repeats":18}]'
SUBSETS_JSON = ""

# --- 进阶云端硬件与架构控制 (已全面同步本地版 v1.3.0 能力) ---
MODEL_VARIANT = "heal"              # 底模分支: "heal"(推荐，数值治愈防溢出) | "base"(官方原版) | "sft"(官方SFT排版对齐版)
VRAM_PRESET = "auto"                # 显存优化档位: "auto"(智能探测) | "6g"(极限4.4G) | "8g"(均衡5.5G) | "12g"(6144流式) | "16g"(8192)
RESOLUTION = 1024                   # 训练目标分辨率: 1024(标准高清) | 896(黄金甜点/8K包可pack2张) | 768(极速) | 512(快速验证)
BASE_QUANT = "auto"                 # 底模权重来源: "auto"(6g/8g自动用int8，12g/16g用全精度) | "int8" | "full"
BASE_PRECISION = "auto"             # 底模计算精度: "auto" | "int8" | "fp8" | "fp8_compute"(L4/A100 Ada+ 自动配16bit dgrad) | "compute"
PRECISION = "auto"                  # 主计算精度: "auto"(T4自动FP16，L4/A100自动BF16) | "fp16" | "bf16"
TUNE = "off"                        # Triton 内核调优模式: "off"(跳过编译耗时) | "table"(首轮自动调优并缓存内核)
OVERLONG = "drop_tokens"            # 超长图文截断保护: "drop_tokens"(默认仅截断超限尾部文本保留全图) | "drop"(整条丢弃)
TEXT_DEVICE = "auto"                # 文本编码器提取设备: "auto"(<=10G/6g走CPU，大显存/大内存智能切换) | "cuda" | "cpu"
GPUS = "auto"                       # 启用 GPU 数量: "auto"(检测到 T4 x2 自动双卡) | "1" | "2"
PARALLEL_MODE = "auto"              # 多卡并行引擎: "auto"(双卡默认 ddp) | "none" | "ddp" | "pipeline"
MAX_TOKENS = 0                      # 覆盖单卡 Token 上限 (0=自动; 双卡 T4 DDP 默认 9216)

# --- 续炼与二阶段微调控制 ---
TRAIN_MODE = "fresh"                # 训练模式: "fresh"(全新训练) | "resume"(断点续炼) | "init_from"(基于已训完的LoRA继续训练)
INIT_LORA_PATH = ""                 # 指定已有 .safetensors 权重路径 (留空则在 init_from/resume 模式下自动检索挂载目录)
SAVE_EVERY = 100                    # 每 N 步自动保存一次中间 checkpoint 权重 (默认 100 步)

# 留空 "" 则全自动扫描 /kaggle/input 下挂载的图集；也可显式指定绝对路径或使用下方 Hugging Face 直拉
DATASET_PATH = ""
DATASET_URL = ""                    # 可选外部图集 ZIP 直拉链接 (支持 | 分隔多镜像容灾)
HF_DATASET_REPO = ""                # 可选 Hugging Face Dataset 仓库 ID (例如 username/my-dataset，填后免上传本地图集)
HF_DATASET_SUBDIR = ""              # 可选 HF 仓库内子目录 (留空下载全仓)
HF_DATASET_REVISION = "main"        # HF 数据集分支或提交
MAX_RUN_MINUTES = 540               # 云端最长安全运行分钟数 (默认 540 分钟 = 9 小时，自动存盘退出防 12 小时强杀；0=关闭)
# ==============================================================================

### 🚀 启动全自动训练循环
点击下方单元格运行，系统将自动拉取项目、下载官方修复版底模、打入兼容补丁并开启训练：

In [ ]:
import os, subprocess, sys

# 1. 获取最新代码
repo_dir = "/kaggle/working/kaggle-moss-lora-pipeline"
if not os.path.exists(repo_dir):
    !git clone https://github.com/abysschaos/kaggle-moss-lora-pipeline.git "{repo_dir}"
else:
    !cd "{repo_dir}" && git pull

%cd "{repo_dir}"

# 2. 执行训练引擎
cmd = [
    sys.executable, "train_moss.py",
    "--lora_name", str(LORA_NAME),
    "--trigger_word", str(TRIGGER_WORD),
    "--repeats", str(REPEATS),
    "--epochs", str(EPOCHS),
    "--steps", str(STEPS),
    "--rank", str(RANK),
    "--alpha", str(ALPHA),
    "--algo", str(ALGO),
    "--lokr_factor", str(LOKR_FACTOR),
    "--input_dropout", str(INPUT_DROPOUT),
    "--rank_dropout", str(RANK_DROPOUT),
    "--module_dropout", str(MODULE_DROPOUT),
    "--target_preset", str(TARGET_PRESET),
    "--loraplus_ratio", str(LORAPLUS_RATIO),
    "--lr", str(LR),
    "--optimizer", str(OPTIMIZER),
    "--lr_scheduler", str(LR_SCHEDULER),
    "--warmup_steps", str(WARMUP_STEPS),
    "--weight_decay", str(WEIGHT_DECAY),
    "--wd_rules", str(WD_RULES),
    "--ema", str(EMA),
    "--grad_acc", str(GRAD_ACC),
    "--timestep", str(TIMESTEP),
    "--timestep_mu", str(TIMESTEP_MU),
    "--text_drop", str(TEXT_DROP),
    "--seed", str(SEED),
    "--save_format", str(SAVE_FORMAT),
    "--variant", str(MODEL_VARIANT),
    "--vram_preset", str(VRAM_PRESET),
    "--resolution", str(RESOLUTION),
    "--base_quant", str(BASE_QUANT),
    "--base_precision", str(BASE_PRECISION),
    "--precision", str(PRECISION),
    "--tune", str(TUNE),
    "--overlong", str(OVERLONG),
    "--text_device", str(TEXT_DEVICE),
    "--gpus", str(GPUS),
    "--parallel", str(PARALLEL_MODE),
    "--max_tokens", str(MAX_TOKENS),
    "--train_mode", str(TRAIN_MODE),
    "--save_every", str(SAVE_EVERY),
    "--max_run_minutes", str(MAX_RUN_MINUTES)
]
if USE_DORA:
    cmd.append("--dora")
if LOKR_FULL_MATRIX:
    cmd.append("--lokr_full_matrix")
if LOKR_DECOMPOSE_BOTH:
    cmd.append("--lokr_decompose_both")
if TRAIN_NORMS:
    cmd.append("--train_norms")
if FLIP:
    cmd.append("--flip")
if SUBSETS_JSON.strip():
    cmd.extend(["--subsets_json", SUBSETS_JSON.strip()])
if INIT_LORA_PATH.strip():
    cmd.extend(["--init_from", INIT_LORA_PATH.strip(), "--resume_from", INIT_LORA_PATH.strip()])
if DATASET_PATH.strip():
    cmd.extend(["--dataset_dir", DATASET_PATH.strip()])
if DATASET_URL.strip():
    cmd.extend(["--dataset_url", DATASET_URL.strip()])
if HF_DATASET_REPO.strip():
    cmd.extend(["--hf_dataset_repo", HF_DATASET_REPO.strip(), "--hf_dataset_revision", str(HF_DATASET_REVISION).strip() or "main"])
    if HF_DATASET_SUBDIR.strip():
        cmd.extend(["--hf_dataset_subdir", HF_DATASET_SUBDIR.strip()])

print(f"Executing: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

### 📦 成果查看与一键打包下载
训练完成后，运行下方单元格查看生成的文件，并直接在右侧面板的 **Output** 区域点击下载：

In [ ]:
import glob, os
print("\n训练产出的 LoRA 权重清单：")
for f in glob.glob(f"/kaggle/working/output/{LORA_NAME}/*"):
    size_mb = os.path.getsize(f) / (1024**2)
    print(f"  -> {os.path.basename(f)} ({size_mb:.2f} MB)")

archive_file = f"/kaggle/working/{LORA_NAME}_lora_results.zip"
if os.path.exists(archive_file):
    print(f"\n✅ 压缩包已准备就绪: {archive_file} ({os.path.getsize(archive_file)/(1024**2):.2f} MB)")
    print("可在右侧 'Output' 下载该 ZIP（内含 ComfyUI/LyCORIS 权重、training_summary.json 与 metrics.jsonl），或在本地 GUI【☁️ 云端遥控】一键拉回！")